<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Distance Metrics & Curse of Dimensionality</h1></center>

Distance metrics define **how similar two points are** — they underpin KNN, K-Means, hierarchical clustering, and SVMs.

### Topics:
1. [Common Distance Metrics](#1)
2. [Unit Ball Visualisation](#2)
3. [Curse of Dimensionality](#3)
4. [Effect of Scaling on KNN](#4)
5. [Choosing the Right Metric](#5)

In [ ]:
import seaborn as sns
dist_colors = ['#03045e', '#0077b6', '#00b4d8', '#90e0ef', '#caf0f8']
print("Distance Metrics Colors:")
sns.palplot(sns.color_palette(dist_colors))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print("Libraries loaded.")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Common Distance Metrics</h1></center>

| Metric | Formula | Best for |
|---|---|---|
| Euclidean (L2) | $\sqrt{\sum(x_i-y_i)^2}$ | Continuous, similar scale |
| Manhattan (L1) | $\sum|x_i-y_i|$ | Robust to outliers, high-d |
| Minkowski | $(\sum|x_i-y_i|^p)^{1/p}$ | Generalises L1 and L2 |
| Chebyshev | $\max|x_i-y_i|$ | p=∞ limit |
| Cosine | $1 - \frac{x\cdot y}{\|x\|\|y\|}$ | Text, sparse, magnitude-free |
| Mahalanobis | $\sqrt{(x-y)^T\Sigma^{-1}(x-y)}$ | Correlated features |

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'],['\\(','\\)']]}});</SCRIPT>

<a id = "2"></a>
<center><h1 style = "background:#03045e ;color:white;border:0;font-weight:bold">Unit Ball Visualisation</h1></center>

The **unit ball** is the set of all points at distance ≤ 1 from the origin. Its shape reveals how each metric measures closeness.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=100)

theta = np.linspace(0, 2*np.pi, 1000)

# L2
axes[0].plot(np.cos(theta), np.sin(theta), color="#03045e", lw=2.5)
axes[0].set_title("L2 (Euclidean) Unit Ball", fontweight="bold")

# L1
axes[1].plot([0, 1, 0, -1, 0], [1, 0, -1, 0, 1], color="#0077b6", lw=2.5)
axes[1].set_title("L1 (Manhattan) Unit Ball", fontweight="bold")

# Linf
axes[2].plot([-1,1,1,-1,-1], [-1,-1,1,1,-1], color="#00b4d8", lw=2.5)
axes[2].set_title("L∞ (Chebyshev) Unit Ball", fontweight="bold")

for ax in axes:
    ax.set_aspect("equal")
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)

plt.tight_layout(); plt.show()

<a id = "3"></a>
<center><h1 style = "background:#0077b6 ;color:white;border:0;font-weight:bold">Curse of Dimensionality</h1></center>

As dimensions increase, all pairwise distances **converge** — points become equidistant, making nearest-neighbour search meaningless.

$$\lim_{d\to\infty} \frac{d_{\max} - d_{\min}}{d_{\min}} \to 0$$

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'],['\\(','\\)']]}});</SCRIPT>

In [ ]:
dimensions = [2, 5, 10, 25, 50, 100, 250, 500, 1000]
ratios, covs = [], []

for d in dimensions:
    pts  = np.random.randn(500, d)
    dmat = cdist(pts[:100], pts[:100], metric="euclidean")
    np.fill_diagonal(dmat, np.nan)
    ratios.append(np.nanmax(dmat) / np.nanmin(dmat))
    covs.append(np.nanstd(dmat) / np.nanmean(dmat))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=100)
axes[0].plot(dimensions, ratios, "o-", color="#03045e")
axes[0].set_xscale("log")
axes[0].set_xlabel("Dimensions (log scale)")
axes[0].set_ylabel("dmax / dmin")
axes[0].set_title("Distance Ratio Collapses with Dimension", fontweight="bold")

axes[1].plot(dimensions, covs, "s-", color="#0077b6")
axes[1].set_xscale("log")
axes[1].set_xlabel("Dimensions (log scale)")
axes[1].set_ylabel("CoV of distances")
axes[1].set_title("Distances Become Similar (Low CoV)", fontweight="bold")

plt.tight_layout(); plt.show()

<a id = "4"></a>
<center><h1 style = "background:#00b4d8 ;color:black;border:0;font-weight:bold">Effect of Scaling on KNN</h1></center>

In [ ]:
X_raw, y_cls = make_classification(n_samples=800, n_features=5, n_informative=3,
                                    n_redundant=0, random_state=42)
# Add very different scales
X_scaled = X_raw.copy()
for i, sf in enumerate([1, 100, 0.01, 50, 1]):
    X_scaled[:, i] *= sf

configs = [
    ("No scaling",     X_scaled, KNeighborsClassifier(n_neighbors=5)),
    ("StandardScaler", X_scaled, make_pipeline(StandardScaler(), KNeighborsClassifier(5))),
]

for label, X_use, pipe in configs:
    scores = cross_val_score(pipe, X_use, y_cls, cv=5, scoring="accuracy")
    print(f"{label:<20}: CV Accuracy = {scores.mean():.4f} ± {scores.std():.4f}")

<a id = "5"></a>
<center><h1 style = "background:#90e0ef ;color:black;border:0;font-weight:bold">Choosing the Right Metric</h1></center>

In [ ]:
# Test metrics on text data
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

cats = ["sci.space", "rec.sport.hockey", "comp.graphics"]
news = fetch_20newsgroups(subset="train", categories=cats,
                          remove=("headers","footers","quotes"))
tfidf = TfidfVectorizer(max_features=500, stop_words="english")
X_text = tfidf.fit_transform(news.data)
y_text = news.target

metric_results = []
for metric in ["euclidean", "manhattan", "cosine"]:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric, n_jobs=-1)
    scores = cross_val_score(knn, X_text, y_text, cv=5, scoring="accuracy")
    metric_results.append({"Metric": metric, "CV Accuracy": scores.mean(),
                            "Std": scores.std()})

res_df = pd.DataFrame(metric_results).round(4)
print(res_df.to_string(index=False))
print("\nCosine usually wins for TF-IDF / high-dimensional sparse data.")

<center><h1 style = "background:#caf0f8 ;color:black;border:0;font-weight:bold">Summary Table</h1></center>

In [ ]:
summary = [
    ["Euclidean (L2)",  "Dense, similar scale", "General purpose"],
    ["Manhattan (L1)",  "Outliers present",     "High-dimensional"],
    ["Cosine",          "Text / sparse",        "Magnitude irrelevant"],
    ["Mahalanobis",     "Correlated features",  "After scaling"],
    ["Chebyshev",       "Grid/game movement",   "Chess-style distance"],
]
pd.DataFrame(summary, columns=["Metric", "Best When", "Notes"])